In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "proyecto_ecommerce")
dbutils.widgets.text("raw_path", "/Volumes/proyecto_ecommerce/raw/raw_files")

catalogo = dbutils.widgets.get("catalogo")
raw_path = dbutils.widgets.get("raw_path")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze")

print(f"Catálogo: {catalogo}")
print(f"Ruta de Raw: {raw_path}")

In [0]:
particiones = sorted([
    f.name for f in dbutils.fs.ls(raw_path)
    if f.name.startswith("ingestion_date=")
])

print(f"Particiones encontradas: {particiones}")

ultima_particion = particiones[-1]
print(f"Última partición: {ultima_particion}")

In [0]:
import shutil
from pathlib import Path

REFERENCIA_PATH = "/Volumes/proyecto_ecommerce/staging/fuente_externa/referencia"
destino_particion_hoy = f"{raw_path}/{ultima_particion}"

for archivo in Path(REFERENCIA_PATH).glob("*.*"):
    if archivo.is_file():
        shutil.copy(archivo, f"{destino_particion_hoy}{archivo.name}")
        print(f"Copiado a Raw: {archivo.name}")

In [0]:
df_clientes_raw = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .csv(f"{raw_path}/{ultima_particion}clientes.csv")
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_fecha_ingesta", F.current_timestamp())
)

df_clientes_raw.printSchema()
display(df_clientes_raw.limit(5))
print(f"Total filas: {df_clientes_raw.count()}")

In [0]:
(df_clientes_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.bronze.clientes"))

print(f"Guardado: {catalogo}.bronze.clientes -> {df_clientes_raw.count()} filas")

In [0]:
df_productos_api = (
    spark.read.option("multiLine", True)
    .json(f"{raw_path}/{ultima_particion}productos.json")
)

df_productos_api.printSchema()

In [0]:
df_productos_raw = (
    df_productos_api
    .select(F.explode("productos").alias("p"), "fuente", "generado_en")
    .select("p.*", "fuente", "generado_en")
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_fecha_ingesta", F.current_timestamp())
)

df_productos_raw.printSchema()
display(df_productos_raw)
print(f"Total filas: {df_productos_raw.count()}")

In [0]:
(df_productos_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.bronze.productos"))

print(f"Guardado: {catalogo}.bronze.productos -> {df_productos_raw.count()} filas")

In [0]:
df_ordenes_raw = (
    spark.read.option("multiLine", True)
    .json(f"{raw_path}/ingestion_date=*/ordenes_*.json")
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_fecha_ingesta", F.current_timestamp())
)

df_ordenes_raw.printSchema()
display(df_ordenes_raw.limit(10))
print(f"Total filas: {df_ordenes_raw.count()}")

In [0]:
display(
    df_ordenes_raw
    .filter(F.col("fecha_hora").rlike("^[0-9]+$"))
    .limit(10)
)

In [0]:
(df_ordenes_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.bronze.ordenes"))

print(f"Guardado: {catalogo}.bronze.ordenes -> {df_ordenes_raw.count()} filas")